In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys
%matplotlib inline
from matplotlib.gridspec import GridSpec
import plotly.offline as py
import plotly.express as px
import plotly.graph_objs as go

import spacy
import re
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

In [5]:
# Get project root (parent of notebooks directory)
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.database.engine import get_db_engine

engine = get_db_engine()

Database connection successful!


In [7]:
df = pd.read_sql("SELECT * FROM Gold.Fact_Orders", engine).drop(columns = ['order_id', 'customer_unique_id',
                                                                           'purchase_date_key', 'load_date_timestamp'])
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   order_status          99441 non-null  object 
 1   total_payment         99440 non-null  float64
 2   primary_payment_type  99440 non-null  object 
 3   review_score          65049 non-null  float64
 4   review_text           26816 non-null  object 
 5   delivery_days_actual  96476 non-null  float64
 6   is_late_delivery      96476 non-null  object 
 7   is_invalid_payment    99441 non-null  bool   
dtypes: bool(1), float64(3), object(4)
memory usage: 5.4+ MB


In [9]:
df = df.dropna(subset = 'review_text').reset_index(drop = True).dropna()
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 25625 entries, 0 to 26815
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   order_status          25625 non-null  object 
 1   total_payment         25625 non-null  float64
 2   primary_payment_type  25625 non-null  object 
 3   review_score          25625 non-null  float64
 4   review_text           25625 non-null  object 
 5   delivery_days_actual  25625 non-null  float64
 6   is_late_delivery      25625 non-null  object 
 7   is_invalid_payment    25625 non-null  bool   
dtypes: bool(1), float64(3), object(4)
memory usage: 1.6+ MB


In [12]:
# Loading Portuguese model
nlp = spacy.load('pt_core_news_lg')

def refine_review(text):
    # Initialization the spaCy document
    doc = nlp(text)
    
    # Keeping only these
    important_tags = ['Noun', 'Adj', 'Verb', 'Adv']
    
    # List comprehensing the processed text
    clean_tokens = [
        token.lemma_.lower()                # Lemmatization
        for token in doc                    # Iterating through tokens
        if not token.is_stop                # Removing stop words
        and token.pos_ in important_tags    # Keeping important POS
        and len(token.text) > 2             # Removing insignifacant noise
    ]
    
    return " ".join(clean_tokens)